# 📘 Episodic + Semantic Memory Stack

Standard chatbot memory lasts one session. A personalised assistant needs memory that persists and is
*structured*. This notebook implements a memory architecture that mirrors human cognition with two kinds of
long-term memory:

1. **Episodic memory** — specific past interactions: "What happened?" (e.g. "Last time, the user asked about
   Apple's stock."). Stored as one-sentence summaries and recalled by similarity.
2. **Semantic memory** — facts the user has stated about themselves: "What do I know?" (e.g.
   `user --IS_A--> conservative investor`). Stored as `(subject, relation, object)` facts and recalled by similarity.

## Learning Objectives
In this notebook, you will learn:
1. **Two memory namespaces** - episodic summaries and semantic facts in one LangGraph store with a vector index
2. **A memory maker** - summarise each turn, and extract facts from the user's own words with structured output
3. **A memory-augmented agent** - a LangGraph flow of retrieve → generate → update
4. **The memory test** - answer a question that only makes sense with both kinds of memory

## Where this fits
Module 03 kept semantic facts
([`03_long_term_semantic_memory_sqlite.ipynb`](02b_long_term_semantic_memory_sqlite.ipynb))
and episodes
([`04_long_term_episodic_memory_sqlite.ipynb`](02c_long_term_episodic_memory_sqlite.ipynb))
in a LangGraph store, recalled by key and by recency. This module looks at research-inspired memory
*architectures*. This first one recalls both kinds of memory by **similarity**, and writes them in a graph node
after every turn instead of through tools. `02_graph_memory_multi_hop.ipynb` stores semantic memory as a graph
and traverses it.

## Prerequisites
- `03_long_term_semantic_memory_sqlite.ipynb` and `04_long_term_episodic_memory_sqlite.ipynb` (module 03): the
  `Store` API and the `Runtime` + `context` pattern
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env` (chat model and embeddings)

| Property | Value |
|---|---|
| Origin | Tulving (1972) episodic vs. semantic memory distinction; modern LLM versions: MemGPT (2023), generative agents (Park et al., 2023) |

## 📖 Definition
An **Episodic + Semantic Memory Stack** is an agent architecture that maintains two types of long-term memory.
**Episodic memory** stores a log of experiences (here, a one-sentence summary of each turn) and is searched by
semantic similarity. **Semantic memory** stores extracted, structured knowledge (here, `(subject, relation, object)`
facts about the user), also searched by similarity. A knowledge graph is another common home for semantic
memory — see `02_graph_memory_multi_hop.ipynb`.

### High-Level Workflow

1.  **Interaction:** The agent has a conversation with the user.
2.  **Memory Retrieval (Recall):** For a new user message, the agent first searches both memories.
    *   It searches the **episodic** index for similar past turns.
    *   It searches the **semantic** index for related facts.
3.  **Augmented Generation:** The retrieved memories are added to the prompt, so the answer is aware of past
    interactions and known facts.
4.  **Memory Creation (Encoding):** A synchronous `update` node runs right after the answer, in the same graph call.
    *   It summarises the turn (the new **episodic** memory).
    *   It extracts facts from the **user's message only** (the new **semantic** memory). The assistant's reply is
        never mined for facts, so the model's own suggestions can't come back later as "things the user said".
        The subject is always `user`, set in code, so "Alex", "I" and "the user" don't become three people.
5.  **Memory Storage:** Both are embedded and written to one store under two namespaces. A fact's key is its
    normalised text, so a restated fact overwrites itself instead of piling up duplicates.

### When to Use
*   **Long-Term Personal Assistants:** An assistant that remembers your preferences, projects, and personal details over weeks or months.
*   **Personalized Systems:** E-commerce bots that remember your style, or educational tutors that remember your learning progress and weak spots.
*   **Complex Research Agents:** An agent that builds up a knowledge base about a topic as it explores documents.

### Strengths & Weaknesses
*   **Strengths:**
    *   **True Personalization:** Enables context and learning that persists indefinitely, far beyond a single session's context window.
    *   **Two kinds of recall:** Episodes answer "what happened last time?", facts answer "what do I know about this user?" — each from a small, focused index.
*   **Weaknesses:**
    *   **Extraction quality:** The extractor decides what counts as a durable fact. A loose prompt stores noise; a strict one misses facts.
    *   **Latency:** Encoding runs inside the request here. Production systems often move it to a background job so the user doesn't wait for it.
    *   **Memory Bloat & Pruning:** Over time, the memory stores can become massive. Strategies for summarizing, consolidating, or pruning old/irrelevant memories are essential for long-term performance.

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library deprecation warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: summarises, extracts facts and answers
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🗄️ Part 1: One Store, Two Memories

Both memories live in one LangGraph `InMemoryStore` with a vector index over OpenAI embeddings. It is the same
`Store` API module 03 used (`put`, `search`), with `index=` switched on so `search(..., query=...)` ranks items by
similarity. Each user gets two namespaces: `(user_id, "episodic")` and `(user_id, "semantic")`.

We use the store rather than a separate vector library (Chroma, FAISS) for two reasons: it keeps one memory API
across the course — a durable store with a vector index, such as `PostgresStore` with pgvector, takes the same
calls — and keyed `put`s make deduplicating facts free.

In [ ]:
# ============================================================================
# PART 1: ONE STORE, TWO NAMESPACES
# ============================================================================
from dataclasses import dataclass

from aied_helpers import get_embeddings
from langgraph.store.memory import InMemoryStore

embeddings = get_embeddings(provider="openai", model="text-embedding-3-small")

# index= turns on semantic search: the "text" field of every value is embedded on put()
store = InMemoryStore(index={"embed": embeddings, "dims": 1536, "fields": ["text"]})


@dataclass
class Context:
    user_id: str  # trusted: set by your server from the login, never by the model


def episodic_ns(user_id: str) -> tuple:
    return (user_id, "episodic")


def semantic_ns(user_id: str) -> tuple:
    return (user_id, "semantic")


def recall(store, namespace: tuple, query: str, k: int) -> list:
    """The k items most similar to `query` (search() returns 10 when no limit is given)."""
    return store.search(namespace, query=query, limit=k)


print("✅ Store ready: episodic and semantic namespaces, embedded with text-embedding-3-small")

---
## ✍️ Part 2: The Memory Maker

After every turn the memory maker writes to both namespaces:
- **Episodic**: a one-sentence summary of the exchange (the user's message and the answer).
- **Semantic**: durable facts the **user** stated about themselves, as `user --RELATION--> object`. The extractor
  sees only the user's message, so questions and the assistant's opinions never become facts, and code — not the
  model — writes the subject. A fact's normalised text is its key, so restating a fact overwrites it.

In [ ]:
# ============================================================================
# PART 2: SUMMARISE THE TURN AND EXTRACT THE USER'S FACTS
# ============================================================================
import re
import uuid

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class UserFact(BaseModel):
    relation: str = Field(description="A short relation in UPPER_SNAKE_CASE, e.g. 'HAS_NAME', 'IS_A', 'INTERESTED_IN'.")
    object: str = Field(description="The value, e.g. 'conservative investor' or 'Alex'.")


class UserFacts(BaseModel):
    """Durable facts the user states about themselves."""
    facts: list[UserFact] = Field(default_factory=list)


summarizer = ChatPromptTemplate.from_messages([
    ("system", "Create a concise, one-sentence summary of the following user-assistant interaction. "
               "It will be used as a memory for future recall."),
    ("human", "Interaction:\n{interaction}"),
]) | llm

# The extractor only ever sees the user's message
extractor = ChatPromptTemplate.from_messages([
    ("system", "Extract durable facts the user states about THEMSELVES in this message: name, preferences, goals, "
               "constraints. A question or a request is not a fact. Return an empty list if there are none."),
    ("human", "User message:\n{user_input}"),
]) | llm.with_structured_output(UserFacts)


def create_memories(store, user_id: str, user_input: str, assistant_output: str) -> None:
    interaction = f"User: {user_input}\nAssistant: {assistant_output}"
    summary = summarizer.invoke({"interaction": interaction}).content
    store.put(episodic_ns(user_id), str(uuid.uuid4()), {"text": summary})
    print(f"  [episodic] {summary}")

    for f in extractor.invoke({"user_input": user_input}).facts:
        relation = re.sub(r"\W+", "_", f.relation).strip("_").upper()
        fact = f"user --{relation}--> {f.object.strip()}"  # the subject is set here, never by the model
        store.put(semantic_ns(user_id), fact.lower(), {"text": fact})  # key = normalised text: restating overwrites
        print(f"  [semantic] {fact}")


print("✅ Memory maker ready")

---
## 🤖 Part 3: The Memory-Augmented Agent

A three-node LangGraph flow: **retrieve** relevant memories from both namespaces, **generate** an answer with
them in the prompt, then **update** memory with the new interaction. The nodes reach the store and the trusted
`user_id` through `Runtime[Context]`, the pattern from module 03.

In [ ]:
# ============================================================================
# PART 3: RETRIEVE -> GENERATE -> UPDATE
# ============================================================================
from typing import Optional

from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from typing_extensions import TypedDict


class AgentState(TypedDict):
    user_input: str
    retrieved_memories: Optional[str]
    generation: str


def retrieve_memory(state: AgentState, runtime: Runtime[Context]) -> dict:
    user_id, query = runtime.context.user_id, state["user_input"]
    # Similarity picks the episodes; time orders them, so "that stock" can resolve to the latest one
    episodes = sorted(recall(runtime.store, episodic_ns(user_id), query, k=2), key=lambda it: it.created_at)
    facts = recall(runtime.store, semantic_ns(user_id), query, k=5)
    context = ("Relevant past conversations (episodic memory, oldest first):\n"
               + ("\n".join(f"- {e.value['text']}" for e in episodes) or "- (none)")
               + "\n\nRelevant facts (semantic memory):\n" + ("\n".join(f"- {f.value['text']}" for f in facts) or "- (none)"))
    print(f"--- Retrieved ---\n{context}")
    return {"retrieved_memories": context}


generator = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful, personalised financial assistant. Use the retrieved memories to tailor "
               "your answer. If the memories state a preference (e.g. a conservative investor), you MUST respect it. "
               "If the question refers back to something ('that stock'), resolve it from the most recent past "
               "conversation. This is an educational demo, not financial advice."),
    ("human", "My question is: {user_input}\n\nMemories that might be relevant:\n{retrieved_memories}"),
]) | llm


def generate_response(state: AgentState) -> dict:
    generation = generator.invoke(state).content
    print(f"--- Answer ---\n{generation}")
    return {"generation": generation}


def update_memory(state: AgentState, runtime: Runtime[Context]) -> dict:
    print("--- Updating memory ---")
    create_memories(runtime.store, runtime.context.user_id, state["user_input"], state["generation"])
    return {}


workflow = StateGraph(AgentState, context_schema=Context)
workflow.add_node("retrieve", retrieve_memory)
workflow.add_node("generate", generate_response)
workflow.add_node("update", update_memory)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "update")
workflow.add_edge("update", END)

memory_agent = workflow.compile(store=store)
print("✅ Memory-augmented agent compiled")

---
## 🧪 Part 4: The Memory Test

Turns 1 and 2 seed the memory. Turn 3 — "Based on my goals, what's a good alternative to that stock?" —
mentions neither the goals nor the stock: answering it needs the **fact** (conservative, established tech)
and the **episode** (the previous question was about Apple).

In [ ]:
# ============================================================================
# PART 4: THREE TURNS
# ============================================================================
USER = Context(user_id="alex")


def fact_count() -> int:
    return len(store.search(semantic_ns(USER.user_id), limit=100))


results, facts_after_turn = [], []
for i, query in enumerate([
    "Hi, my name is Alex. I'm a conservative investor, and I'm mainly interested in established tech companies.",
    "What do you think about Apple (AAPL)?",
    "Based on my goals, what's a good alternative to that stock?",
], 1):
    print(f"\n================ TURN {i}: {query}")
    results.append(memory_agent.invoke({"user_input": query}, context=USER))
    facts_after_turn.append(fact_count())

Read the "Retrieved" block of turn 3: it should list the Apple episode and the conservative-investor facts,
and the answer should suggest an established tech company other than Apple while respecting the conservative
goal. Neither was in the turn-3 question itself. The next cell lists both namespaces and checks the recall in code.

In [ ]:
# ============================================================================
# INSPECT BOTH NAMESPACES AND CHECK THE RECALL
# ============================================================================
episodes = sorted(store.search(episodic_ns(USER.user_id), limit=100), key=lambda it: it.created_at)
facts = sorted(it.value["text"] for it in store.search(semantic_ns(USER.user_id), limit=100))

print(f"Episodic memory ({len(episodes)} episodes, oldest first):")
for it in episodes:
    print("  -", it.value["text"])
print(f"\nSemantic memory ({len(facts)} facts):")
for f in facts:
    print("  -", f)

recalled_episodes, recalled_facts = results[2]["retrieved_memories"].lower().split("relevant facts")
checks = {
    "turn 3 recalled the Apple episode": "apple" in recalled_episodes,
    "turn 3 recalled the conservative-investor fact": "conservative" in recalled_facts,
    f"the two questions added no facts (facts after each turn: {facts_after_turn})":
        facts_after_turn[0] == facts_after_turn[-1] > 0,
}
print()
for name, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {name}")

Compare the semantic namespace with what the user said in turn 1: only facts about the user, all with the
subject `user`, and nothing taken from the assistant's answers. The two questions (turns 2 and 3) added no facts.
They live on as episodes instead, which is what let turn 3 resolve "that stock".

---
## 📝 Summary

In this notebook, we learned:

### 1. Two memory namespaces
- Episodic memory holds summaries of past turns; semantic memory holds extracted facts — two namespaces of one `InMemoryStore` with a vector index, both searched by similarity

### 2. A memory maker
- Each turn is summarised; facts come from the user's own words only, with the subject fixed in code and the fact text as its key

### 3. A memory-augmented agent
- Retrieve → generate → update, as a three-node LangGraph flow whose nodes reach the store through `Runtime[Context]`

### 4. The memory test
- A follow-up that names neither the goal nor the stock was answered from a recalled fact and a recalled episode

Pruning and consolidating both memories becomes essential as they grow.

### Next Steps
- **`02_graph_memory_multi_hop.ipynb`**: store semantic memory as a graph and answer multi-hop questions by traversal